# Notebook 03: Explain & Optimise — Debutanizer Soft Sensor
Stage 8 (SHAP + importance) and Stage 9 (Genetic Algorithm optimisation).


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src.data_io import load_training_data
from src.features import build_causal_features, build_autoregressive_features
from src.evaluate import chronological_split, scale_features
from src.models import run_experiment, get_model_grids
from src.explain import run_explain
from src.optimise import run_ga_optimisation

plt.rcParams["figure.dpi"] = 110


## Load Data and Rebuild Experiments

In [ ]:
df = load_training_data()
# Re-run or load from disk
# NOTE: If model_results.csv already exists, re-running will overwrite it.
# Load pre-run results and rebuild experiment objects for explain/optimise.
from src.models import run_experiment
experiments = run_experiment(df)


## Stage 8: Explainability

We focus on the best causal and best autoregressive models separately.

In [ ]:
from src.explain import explain_best_model_from_experiments
explain_results = explain_best_model_from_experiments(experiments, save_dir="../figures")


In [ ]:
# Display SHAP summary plots
for variant_key in ["causal_lag16", "autoregressive_lag16"]:
    if variant_key in experiments:
        df_res = pd.read_csv("../results/model_results.csv")
        subset = df_res[df_res["feature_variant"] == variant_key &
                        ~df_res["model_name"].str.startswith("Persistence")]
        if not subset.empty:
            best_name = subset.sort_values("test_rmse").iloc[0]["model_name"]
            for plot_type in ["shap_summary", "shap_bar", "grouped_importance"]:
                fpath = f"../figures/{plot_type}_{best_name}_{variant_key}.png"
                if os.path.exists(fpath):
                    print(f"\n{plot_type} — {best_name} on {variant_key}")
                    display(Image(fpath, width=850))


## Stage 9: Genetic Algorithm Optimisation

We use the best **causal** model only. An operator can adjust process variables (u1-u7) in
real time but cannot control past laboratory assay values, so the autoregressive model is
not suitable for optimisation.

**Important caveat:** The causal model achieves Test R² ≈ 0.45. The GA result is indicative
only and must be validated through Aspen Plus simulation before any plant trial.


In [ ]:
# Identify the best causal model and variant
df_res = pd.read_csv("../results/model_results.csv")
causal_ml = df_res[
    df_res["feature_variant"].str.startswith("causal") &
    ~df_res["model_name"].str.startswith("Persistence")
].sort_values("test_rmse")

best_causal_row = causal_ml.iloc[0]
print(f"Best causal model: {best_causal_row['model_name']} on {best_causal_row['feature_variant']}")
print(f"Test RMSE: {best_causal_row['test_rmse']:.4f}, Test R²: {best_causal_row['test_r2']:.4f}")


In [ ]:
# Run GA
best_variant = best_causal_row["feature_variant"]
max_lag = 16 if "lag16" in best_variant else 4
X_full, y_full = build_causal_features(df, max_lag=max_lag)
X_train, X_val, X_test, y_train, y_val, y_test = chronological_split(X_full, y_full)
X_train_s, X_val_s, X_test_s, scaler = scale_features(X_train, X_val, X_test)

best_model = experiments[best_variant]["models"][best_causal_row["model_name"]]["model"]
input_cols = ["u1","u2","u3","u4","u5","u6","u7"]
X_train_raw = df.iloc[:len(X_train)][input_cols]

ga_results = run_ga_optimisation(
    causal_model=best_model,
    scaler=scaler,
    X_train_s=X_train_s,
    y_train=y_train,
    X_train_raw=X_train_raw,
    feature_columns=list(X_full.columns),
    max_lag=max_lag,
    input_cols=input_cols,
    n_seeds=5, pop_size=60, n_gen=80,
    save_dir="../figures", results_dir="../results"
)


In [ ]:
# Show convergence plot
display(Image("../figures/ga_convergence.png", width=850))
print("\nOptimisation Summary:")
print(f"Best C4: {ga_results['stability']['best_c4']:.4f}")
print(f"Mean C4: {ga_results['stability']['mean_c4']:.4f}")
print(f"Std C4:  {ga_results['stability']['std_c4']:.4f}")
print(f"\nOptimal operating point (normalised):")
for col, val in zip(input_cols, ga_results["global_best_individual"]):
    print(f"  {col} = {val:.4f}")


## Results Files
- `results/model_results.csv` — all model metrics
- `results/model_comparison.md` — ranked tables and plain-English analysis
- `results/optimisation_summary.md` — GA configuration, results, and limitations
- `results/optimum_for_aspen.csv` — normalised optimum to hand over to Aspen teammate
- `figures/ga_convergence.png` — convergence across 5 seeds
